# AIDA 1141 Final Project – 06 Single Test Evaluation

**Data version 2** (`shared/synthetic-data/`), using the employee split fixed in notebook 01, the model and thresholds saved in notebook 02, and the review policy frozen in notebook 04. Nothing is refitted, no threshold changes, and the policy choice is not reopened.

**Question:** How does the frozen early-warning policy perform on employees it has never seen, and what does that mean for the coordinator's review work?

**What is frozen.** Policy (a): flag an early-warning assessment when the task's start date is **8 or fewer business days** away; notebook 02 chose 8 on training data, and notebook 04 chose this policy on validation. The comparison model (logistic regression) and its full-precision threshold, the "never at risk" reference, the repair-check definition (notebook 05) and the test split are also fixed. The notebook first checks every input against the fingerprints saved with the frozen policy and stops if any has changed.

**What this notebook reports**, separately from validation (validation appears only as context):

- **Population:** test employees, tasks and early-warning assessments by checkpoint; overdue assessments (already past the deadline, reported as the separate urgent list); assessments whose outcome was unknown at the extract (excluded and counted). Task-level results use the tasks in the scored early-warning population as their denominator, not every task with a test assessment, and the employees represented in scoring are reported separately from all test employees.
- **Assessment level:** recall (the primary measure), precision, flags and the share flagged, and a coordinator view per 100 assessments.
- **Task level:** review volume (unique tasks flagged), at-risk tasks **flagged at least once**, and at-risk tasks **flagged before their start date** — the timing of a flag is checked from its timestamp, so eventual coverage and timely warning are reported separately.
- **Missed warnings:** every at-risk assessment the policy did not flag, with whether a later assessment of the same task was flagged and how many business days remained when it was.
- **Groups:** results by employee type, department, task type, intake channel and checkpoint, labelled with notebook 04's three categories, read from the frozen policy: **limited recall support** (fewer than 30 at-risk assessments); **documented concern** (recall at least 10 percentage points below the policy's own overall recall, or an at-risk task with no early-warning flag at any checkpoint); **serious concern** (at least 30 at-risk assessments and recall below 80%).
- **Comparison methods:** the model and "never at risk" on the same measures, with ranking measures; they are reported, not chosen.
- **Cleaning repairs:** notebook 05's procedure applied unchanged, including its audit of the rule's inputs.

A missing measure is reported as missing (for example precision when nothing is flagged), never as zero. The results are saved in `test_results.json` with the input fingerprints (including those of the repair-check definition, the generator and the frozen policy file), the thresholds and the population counts; reruns compare the results with the saved record and stop if they differ. A poor test result is reported and carried into the decision brief; it does not change the policy here.

All records are synthetic: the rates describe the generator's assumptions, not real employees or real HR performance.

In [1]:
import hashlib
import importlib.util
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score

DATA_VERSION = "2"
SD = Path("../../../shared/synthetic-data")          # data version 2
OUTPUT_DIR = Path("../outputs")

read = lambda path: pd.read_csv(path, dtype=str, keep_default_na=False)
sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
settings = json.loads((OUTPUT_DIR / "model_settings.json").read_text())          # notebook 02
policy = json.loads((OUTPUT_DIR / "review_policy.json").read_text())             # notebook 04 (frozen)
repair_definition = json.loads((OUTPUT_DIR / "repair_check_definition.json").read_text())   # notebook 05 (protected)
requests = read(SD / "cleaning/output/requests_cleaned.csv")
raw_requests = read(SD / "requests/data/requests.csv")
assessments = read(SD / "requests/data/assessments.csv")
events = read(SD / "requests/data/status_events.csv")
employee_split = read(OUTPUT_DIR / "employee_split.csv")
feature_list = read(OUTPUT_DIR / "feature_list.csv")
split_summary_01 = pd.read_csv(OUTPUT_DIR / "split_summary.csv", index_col="split")
cleaning_info = json.loads((SD / "cleaning/output/requests_cleaned_info.json").read_text())

# Every input must be the one the frozen policy was chosen with
FILES = {"requests_cleaned.csv": SD / "cleaning/output/requests_cleaned.csv",
         "assessments.csv": SD / "requests/data/assessments.csv",
         "employee_split.csv": OUTPUT_DIR / "employee_split.csv", "feature_list.csv": OUTPUT_DIR / "feature_list.csv",
         "model_settings.json": OUTPUT_DIR / "model_settings.json", "model.joblib": OUTPUT_DIR / "model.joblib",
         "validation_predictions.csv": OUTPUT_DIR / "validation_predictions.csv"}
FINGERPRINTS = {n: sha(p) for n, p in FILES.items()}
assert FINGERPRINTS == policy["fingerprints_sha256"], "An input differs from the fingerprints saved with the frozen policy (notebook 04)"
assert policy["chosen_policy"] == "a" and not policy["model_used"], "The frozen policy is not the days rule"
assert sha(SD / "requests/data/requests.csv") == cleaning_info["source_requests_sha256"] \
    and sha(SD / "requests/data/status_events.csv") == cleaning_info["source_events_sha256"], \
    "The raw export or event log differs from the ones the cleaned file was made from"
assert repair_definition["applies_to"].startswith("validation here; notebooks 06-07 apply the same rule unchanged")
# Further fingerprints for the test-run record. No earlier artifact of this project records hashes of these three files,
# so there is no expected value to compare with here; reruns compare them with the saved test record instead.
RUN_FINGERPRINTS = {**FINGERPRINTS, "review_policy.json": sha(OUTPUT_DIR / "review_policy.json"),
                    "repair_check_definition.json": sha(OUTPUT_DIR / "repair_check_definition.json"),
                    "generate_requests.py": sha(SD / "requests/generate_requests.py")}
assert sklearn.__version__ == settings["package_versions"]["scikit-learn"], "scikit-learn differs from the version the model was saved with"

model = joblib.load(OUTPUT_DIR / "model.joblib")
FEATURES = settings["features"]
NUMERIC = feature_list.loc[feature_list["type"] == "numeric", "feature"].tolist()
DAYS_THRESHOLD = policy["threshold"]
MODEL_THRESHOLD = settings["model_threshold"]
assert DAYS_THRESHOLD == settings["days_threshold"] and policy["flag_rule"] == f"business_days_until_start <= {DAYS_THRESHOLD}"
# Notebook 04's group rules, read from the frozen policy so test and validation use the same categories
criteria = policy["decision_criteria"]
GROUP_RULES = {"limited_support_below_at_risk": criteria["limited_recall_support_below_at_risk"],
               "documented_concern_gap_pp": criteria["documented_concern_gap_pp_vs_own_overall"],
               "serious_concern_recall_below_pct": criteria["serious_concern_recall_below_pct"]}
assert GROUP_RULES == {"limited_support_below_at_risk": 30, "documented_concern_gap_pp": -10, "serious_concern_recall_below_pct": 80}

# Business-day calendar from the data version 2 generator
spec = importlib.util.spec_from_file_location("generator", SD / "requests/generate_requests.py")
generator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(generator)

# Notebook 02's input table, rebuilt for every split; the model must still reproduce its validation scores
data = (assessments
        .merge(requests[["request_id", "employee_id", "task_type", "hiring_department", "employee_type", "intake_channel",
                         "info_issue_at_intake"]].rename(columns={"employee_id": "request_employee_id"}),
               on="request_id", how="left", validate="many_to_one")
        .merge(employee_split[["employee_id", "split"]], on="employee_id", how="left", validate="many_to_one"))
data["intake_channel"] = data["intake_channel"].replace("", "unknown")   # missing at source (notebook 01)
data[NUMERIC] = data[NUMERIC].astype(int)
data["assessed_at"] = pd.to_datetime(data["assessed_at"], format="%Y-%m-%dT%H:%M")
validation_02 = pd.read_csv(OUTPUT_DIR / "validation_predictions.csv")
val_rows = data.set_index("assessment_id").loc[validation_02["assessment_id"]]
MODEL_REPRODUCED = bool(np.allclose(model.predict_proba(val_rows[FEATURES])[:, 1], validation_02["risk_probability"], rtol=0, atol=1e-12))
assert MODEL_REPRODUCED, "The saved model does not reproduce notebook 02's validation scores"

print(f"Data version {DATA_VERSION}; scikit-learn {sklearn.__version__}; inputs match the frozen policy's fingerprints")
print(f"Frozen policy (a): {policy['flag_rule']}; comparison model threshold {MODEL_THRESHOLD:.4f} (full precision saved)")
print(f"The saved model reproduces notebook 02's {len(validation_02):,} validation scores")
print(f"Group rules from the frozen policy: {GROUP_RULES}")

Data version 2; scikit-learn 1.9.1; inputs match the frozen policy's fingerprints
Frozen policy (a): business_days_until_start <= 8; comparison model threshold 0.1338 (full precision saved)
The saved model reproduces notebook 02's 1,033 validation scores
Group rules from the frozen policy: {'limited_support_below_at_risk': 30, 'documented_concern_gap_pp': -10, 'serious_concern_recall_below_pct': 80}


## A. The test population

The test split holds the employees notebook 01 set aside; none of them was used to fit the model, choose a threshold or choose the policy. The cell below rebuilds the test assessments with notebook 01's rules and checks the counts against notebook 01's split summary before any prediction or result is calculated:

- **Early-warning assessments** (outcome known, made before the task's deadline) are the scored population.
- **Overdue assessments** (made at or after the deadline) are already late; under the policy they go to the separate urgent-review list and are reported separately.
- **Unknown-outcome assessments** (task unfinished at the extract with its deadline still ahead) are excluded from scoring and counted.

The population table gives assessments, tasks and employees for each group and checkpoint, and the share of early-warning assessments labelled at risk, which notebook 01 already reported for the split.

In [2]:
test = data[data["split"] == "test"].copy()
early = test[(test["at_risk"] != "") & (test["past_deadline_at_assessment"] == "False")].copy()
early["y"] = (early["at_risk"] == "True").astype(int)
overdue = test[(test["at_risk"] != "") & (test["past_deadline_at_assessment"] == "True")].copy()
unknown = test[test["at_risk"] == ""].copy()

other_ids = set(data.loc[data["split"] != "test", "employee_id"])
s01 = split_summary_01.loc["test"]
POPULATION_OK = bool(
    len(early) + len(overdue) + len(unknown) == len(test)
    and not set(test["employee_id"]) & other_ids
    and (test["employee_id"] == test["request_employee_id"]).all()
    and test["employee_id"].nunique() == s01["employees"]
    and len(early) == s01["early_warning_assessments"] and int(early["y"].sum()) == s01["early_warning_at_risk"]
    and len(overdue) == s01["overdue_assessments"] and len(unknown) == s01["unknown_outcome_assessments"]
    and early[FEATURES].notna().all().all() and early[FEATURES].ne("").all().all()
    and early["assessment_id"].is_unique)
assert POPULATION_OK, "The rebuilt test population does not match notebook 01"


def population_rows(frame, group, labelled):
    rows = []
    for cp, g in [("all", frame)] + sorted(frame.groupby("checkpoint")):
        rows.append({"group": group, "checkpoint": cp, "assessments": len(g), "tasks": g["request_id"].nunique(),
                     "employees": g["employee_id"].nunique(),
                     "at_risk": int((g["at_risk"] == "True").sum()) if labelled else pd.NA})
    return rows


population = pd.DataFrame(population_rows(early, "early warning (scored)", True) + population_rows(overdue, "overdue (urgent list)", True)
                          + population_rows(unknown, "unknown outcome (excluded)", False))
population["at_risk"] = population["at_risk"].astype("Int64")
population["at_risk_%"] = (100 * population["at_risk"].astype(float) / population["assessments"]).where(
    population["group"] == "early warning (scored)")
population.insert(0, "data_version", DATA_VERSION)
population.to_csv(OUTPUT_DIR / "test_population.csv", index=False)
SCORED_TASKS, SCORED_EMPLOYEES = early["request_id"].nunique(), early["employee_id"].nunique()
print(f"Test split: {test['employee_id'].nunique()} employees and {test['request_id'].nunique()} tasks with any assessment; "
      f"counts match notebook 01")
print(f"Scored early-warning population: {len(early):,} assessments ({int(early['y'].sum())} at risk) on {SCORED_TASKS} tasks "
      f"from {SCORED_EMPLOYEES} employees (the task-level denominator); also {len(overdue)} overdue and {len(unknown)} unknown-outcome assessments")
display(population.drop(columns="data_version").round(1))

Test split: 215 employees and 877 tasks with any assessment; counts match notebook 01
Scored early-warning population: 1,056 assessments (192 at risk) on 829 tasks from 204 employees (the task-level denominator); also 41 overdue and 7 unknown-outcome assessments


,group,checkpoint,assessments,tasks,employees,at_risk,at_risk_%
0,early warning (scored),all,1056,829,204,192,18.2
1,early warning (scored),checkpoint_1,829,829,204,146,17.6
2,early warning (scored),checkpoint_2,227,227,103,46,20.3
3,overdue (urgent list),all,41,41,12,41,NaN
4,overdue (urgent list),checkpoint_1,41,41,12,41,NaN
5,unknown outcome (excluded),all,7,7,3,<NA>,NaN
6,unknown outcome (excluded),checkpoint_1,7,7,3,<NA>,NaN
